In [ ]:
'''
Runs all BAI experiments for AlphaFold protein residue disorder rate (finite-population).

Remarks:
1. PAI is by definition sequential.
2. Because the ML model is fixed, SAI will be run in batch mode (i.e., BAI -- no budget-stabilization).
3. We'll use Zrnic & Candes's f_t(x) = p_t(x) and u_t(x) = 2\min\{ p_t(x), 1 - p_t(x) \} with tau=0.5.
'''
import torch
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
import os, csv

# set a device + specify number of seeds, uniform-mixing & CI target coverage + spread of budgets
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, TAU, ALPHA = 1000, 0.5, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)

# filename for this simulation
fname = f"alphafold_bai.csv"

# dataframe to store all of our results
columns = ["dataset", "budget", "tau", "method", "seed", "thetahat", "coverage", "width", "n_labels"]
if fname not in os.listdir("../logs"):
    with open(f"../logs/{fname}", "wt") as file:
        file.write(",".join(columns) + "\n")

In [ ]:
#### LOAD IN OUR DATA

# for alphafold, just need y_test and fhat_test (fixed AlphaFold-derived ML predictions)
fxs = torch.load("../datasets/alphafold/fhat_test.pt").to(device) # (10802,)
ys = torch.load("../datasets/alphafold/y_test.pt").to(device) # (10802,)

# get our finite-population size
N = ys.shape[0]

# can precompute our uncertainty scores: u_t \propto ( p_t(x), 1 - p_t(x) ) + expectation because fixed.
u_ts = torch.minimum(torch.tensor(1.0, device=device) - fxs, fxs) # (10802,)
Eu_t = u_ts.mean() #(,)

In [ ]:
####  BATCH ACTIVE INFERENCE SAMPLING

# iterate through our budgets
for budget_idx, budget in enumerate(BUDGET_PROPS):
    
    # what is our (expected) budget
    n = int(budget * N)
    
    #### initialize data structures
    
    # set a seed for reproducibility
    torch.random.manual_seed(858 + budget_idx)

    # how many labels have we used up already
    obs_counts = torch.zeros(size=(N_SEEDS, 1), dtype=torch.float32, device=device)

    # recurring sum (WITHOUT NORMALIZATION) of thetahat (need div by N) and sigmahat^2 (need div by N^2)
    thetahats_unnormalized = torch.zeros(size=(N_SEEDS, 1), device=device, dtype=torch.float32)
    sigmahats_sq_unnormalized = torch.zeros(size=(N_SEEDS, 1), device=device, dtype=torch.float32)
    
    #### batch active inference (need loop thru full dataset, no budget stabilization.)
    for t in tqdm(range(N), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}"):
    
        # what are our label probabilities for this current z_t? (Eqn 8 + Algorithm 2, Zrnic & Candes '24)
        eta_t = n / (N * Eu_t) # (,)
        pi_t = (eta_t * u_ts[t]).clamp(min=0.0, max=1.0)

        # mixing with uniform -- Section 7.2, Zrnic & Candes '24
        pi_t_tau = ((1.0 - TAU) * pi_t) + (TAU * (n / N)) # (,)

        # are we going to query this data point ir not?
        xi_t = torch.bernoulli(torch.tile(pi_t_tau, (N_SEEDS, 1))) # (N_SEEDS, 1)

        #### PART 2. UPDATING BATCH ACTIVE INFERENCE ESTIMATOR + BOOKKEEPING

        # update our means and variances
        thetahats_unnormalized += ((xi_t / pi_t_tau) * (ys[t] - fxs[t])) + fxs[t]
        sigmahats_sq_unnormalized += (xi_t * (1.0 - pi_t_tau) / (pi_t_tau ** 2) ) * ((ys[t] - fxs[t]) ** 2)

        # did we exhaust budget? also, did we hit a big-residual datapoint?
        obs_counts += xi_t

    #### COMPUTING BATCH ACTIVE INFERENCE CONFIDENCE INTERVAL AND COVERAGE
    
    # dummy tensor for brevity
    ZERO = torch.tensor(0.0, device=device)

    # get our mean estimate + variance estimate
    thetahats = thetahats_unnormalized / N
    sigmahats = torch.sqrt(torch.maximum(sigmahats_sq_unnormalized, ZERO) / N)
    sigmahats_bar = sigmahats / torch.sqrt(torch.tensor(N, device=device))

    # get our z-score + the lower/upper limits of our interval
    z_score = norm.ppf(1 - (ALPHA / 2))
    lbs = thetahats - z_score * sigmahats_bar
    ubs = thetahats + z_score * sigmahats_bar
    widths = (ubs - lbs).cpu().numpy()
    coverages = ((lbs <= ys.mean(dim=-1, keepdim=True)) & (ys.mean(dim=-1, keepdim=True) <= ubs)).cpu().numpy()
    
    # for file writing purposes, convert thetahats + obs_counts to numpy
    thetahats = thetahats.cpu().numpy()
    obs_counts = obs_counts.cpu().numpy()
    
    # format results + directly write to file
    results = [
        ["alphafold", budget, TAU, "bai", seed, 
         thetahats[seed][0], coverages[seed][0], widths[seed][0], obs_counts[seed][0]
        ] for seed in range(N_SEEDS)]
    with open(f"../logs/{fname}", "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerows(results)